# 03 · Camada Silver


Converte o dado da Bronze em tabelas limpas, tipadas e padronizadas.

**Tratamentos aplicados**

| Problema na Bronze | Tratamento |
| --- | --- |
| Decimal com vírgula e ponto de milhar | Remoção do milhar, troca da vírgula, `CAST` para `DECIMAL(18,2)` |
| Separador de milhar em ano, código e CNAE das desonerações | Remoção do ponto antes do `CAST` |
| Acentos inconsistentes em tipo e finalidade das desonerações | Maiúsculas sem acento (`translate`) |
| CNAE sem zero à esquerda | `lpad` até o tamanho do nível |
| Código e descrição no mesmo campo | `regexp_extract` e `regexp_replace` |
| Código `0` com nome "SEM CNAE" | Coluna `sem_cnae` |
| Ano e mês separados | Coluna `data_ref` |
| Dois níveis de granularidade nas desonerações | Coluna `nivel_agregacao` |
| Grafias divergentes de município entre SEFAZ e IBGE | Tabela de-para por chave normalizada |

**Convenções**

- Nomes de coluna em minúsculas, sem acento.
- CNAE e código IBGE de município com 7 dígitos; COREDE em maiúsculas.
- Colunas de linhagem herdadas da Bronze, mais `_processamento_ts`.


In [0]:
%sql
USE CATALOG mvp_pipeline_vf;

## 1. ICMS por CNAE subclasse

Grão: ano × mês × versão da CNAE × subclasse × nome.

A fonte publica CNAE 1.1 e 2.0 lado a lado e, desde nov/2024, duas categorias residuais sob o código
`0000000`. Por isso versão e nome integram o grão.

In [0]:
%sql
CREATE OR REPLACE TABLE silver.icms_cnae_subclasse AS
SELECT
  CAST(ano AS INT)                                        AS ano,
  CAST(mes AS INT)                                        AS mes,
  make_date(CAST(ano AS INT), CAST(mes AS INT), 1)        AS data_ref,
  lpad(regexp_replace(cod_subclasse, '\\D', ''), 7, '0')  AS cnae_subclasse,
  lpad(regexp_replace(cod_subclasse, '\\D', ''), 7, '0') = '0000000' AS sem_cnae,
  trim(nome_subclasse)                                    AS nome_cnae_subclasse,
  cod_versao                                              AS versao_cnae,
  CAST(replace(replace(valor, '.', ''), ',', '.') AS DECIMAL(18,2)) AS valor_icms,
  _ingestao_ts,
  _fonte,
  current_timestamp()                                     AS _processamento_ts
FROM bronze.icms_cnae_subclasse;

In [0]:
%sql
-- Verificação: quanto do ICMS não tem atividade econômica atribuída
SELECT sem_cnae, count(*) AS linhas, count(DISTINCT versao_cnae) AS versoes,
       round(sum(valor_icms)/1e9, 2) AS valor_bi
FROM silver.icms_cnae_subclasse GROUP BY sem_cnae;

## 2. Arrecadação por município e COREDE

Grão: ano × mês × município × tributo. Cobre ICMS, IPVA e ITCD.


In [0]:
%sql
CREATE OR REPLACE TABLE silver.arrecadacao_municipio AS
SELECT
  CAST(ano AS INT)                                 AS ano,
  CAST(mes AS INT)                                 AS mes,
  make_date(CAST(ano AS INT), CAST(mes AS INT), 1) AS data_ref,
  CAST(cod_corede AS INT)                          AS cod_corede,
  upper(trim(nome_corede))                         AS nome_corede,
  CAST(cod_munic AS INT)                           AS cod_munic_sefaz,
  trim(nome_munic)                                 AS nome_municipio,
  upper(trim(sigla_tipo_arr))                      AS tributo,
  CAST(replace(replace(valor, '.', ''), ',', '.') AS DECIMAL(18,2)) AS valor_arrecadado,
  _ingestao_ts,
  _fonte,
  current_timestamp()                              AS _processamento_ts
FROM bronze.arrecadacao_municipio_corede;

In [0]:
%sql
-- Verificação
SELECT tributo, count(*) AS linhas, min(ano) AS ano_min, max(ano) AS ano_max,
       count(DISTINCT cod_munic_sefaz) AS municipios
FROM silver.arrecadacao_municipio
GROUP BY tributo ORDER BY tributo;

## 3. Desonerações fiscais (2016 a 2025)

Grão: ano × dispositivo × COREDE × subclasse.

| Tratamento | Motivo |
| --- | --- |
| Remoção do ponto em ano, código e CNAE | A fonte grava `2.016`, `1.003`, `4.623.199` |
| Tipo e finalidade sem acento | A fonte alterna `ISENÇÃO` e `ISENCAO`; as regras da Gold comparam texto |
| `lpad` na hierarquia CNAE | Seções A e B sem zero à esquerda |
| `nivel_agregacao` | Simples Nacional, Simples Gaúcho, IPVA e ITCD vêm sem CNAE e COREDE (`SEM COREDE`) |

In [0]:
%sql
CREATE OR REPLACE TABLE silver.desoneracoes AS
SELECT
  CAST(replace(ano, '.', '') AS INT)                  AS ano,
  upper(trim(imposto))                                AS imposto,
  upper(trim(translate(tipo_benef,
        'ÁÀÂÃÉÊÍÓÔÕÚÇáàâãéêíóôõúç', 'AAAAEEIOOOUCaaaaeeiooouc'))) AS tipo_beneficio,
  CAST(replace(cod_benef, '.', '') AS INT)            AS cod_beneficio,
  trim(descr_benef)                                   AS descr_beneficio,
  trim(legislacao_aplicada)                           AS legislacao,
  upper(trim(translate(finalidade,
        'ÁÀÂÃÉÊÍÓÔÕÚÇáàâãéêíóôõúç', 'AAAAEEIOOOUCaaaaeeiooouc'))) AS finalidade,
  trim(justificativa)                                 AS justificativa,
  upper(trim(corede))                                 AS nome_corede,
  trim(cnae_1)                                        AS cnae_secao,
  lpad(regexp_replace(cnae_2, '\\D', ''), 2, '0')     AS cnae_divisao,
  lpad(regexp_replace(cnae_3, '\\D', ''), 3, '0')     AS cnae_grupo,
  lpad(regexp_replace(cnae_5, '\\D', ''), 5, '0')     AS cnae_classe,
  lpad(regexp_replace(cnae_7, '\\D', ''), 7, '0')     AS cnae_subclasse,
  trim(descr_cnae_7)                                  AS nome_cnae_subclasse,
  CAST(replace(replace(vlr_desoner, '.', ''), ',', '.') AS DECIMAL(18,2)) AS valor_desonerado,
  CAST(qtd_cnpj8 AS INT)                              AS qtd_empresas,
  CASE WHEN upper(trim(corede)) = 'SEM COREDE' THEN 'AGREGADO_ESTADUAL'
       ELSE 'DETALHADO' END                           AS nivel_agregacao,
  _ingestao_ts,
  _fonte,
  current_timestamp()                                 AS _processamento_ts
FROM bronze.desoneracoes;

In [0]:
%sql
-- Verificação: volume e cobertura por ano
SELECT ano, count(*) AS linhas, sum(valor_desonerado) AS valor_total,
       count(DISTINCT nome_corede) AS coredes,
       count(DISTINCT cnae_subclasse) AS subclasses
FROM silver.desoneracoes
GROUP BY ano ORDER BY ano;

In [0]:
%sql
-- Verificação: peso de cada nível de granularidade
SELECT nivel_agregacao, count(*) AS linhas,
       sum(valor_desonerado) AS valor,
       round(100 * sum(valor_desonerado) / sum(sum(valor_desonerado)) OVER (), 1) AS perc,
       count(DISTINCT nome_corede) AS coredes,
       count(DISTINCT cnae_subclasse) AS subclasses
FROM silver.desoneracoes
GROUP BY nivel_agregacao;

In [0]:
%sql
-- Verificação: peso de cada nível de granularidade (esperado 480 linhas no agregado)
SELECT nivel_agregacao, count(*) AS linhas,
       sum(valor_desonerado) AS valor,
       round(100 * sum(valor_desonerado) / sum(sum(valor_desonerado)) OVER (), 1) AS perc,
       count(DISTINCT nome_corede) AS coredes,
       count(DISTINCT cnae_subclasse) AS subclasses
FROM silver.desoneracoes
GROUP BY nivel_agregacao;

In [0]:
%sql
-- Verificação: domínios normalizados e hierarquia CNAE com zero à esquerda
SELECT
  (SELECT count(DISTINCT tipo_beneficio) FROM silver.desoneracoes)          AS tipos,
  (SELECT count(DISTINCT finalidade)     FROM silver.desoneracoes)          AS finalidades,
  (SELECT count(*) FROM silver.desoneracoes
   WHERE nivel_agregacao = 'DETALHADO' AND length(cnae_divisao) <> 2)      AS divisao_fora_do_padrao,
  (SELECT count(*) FROM silver.desoneracoes
   WHERE nivel_agregacao = 'DETALHADO'
     AND substr(cnae_subclasse, 1, 5) <> cnae_classe)                      AS subclasse_fora_da_classe;

## 4. Cadastro de contribuintes por setor

Grão: ano × mês × categoria × subclasse.

- Código e descrição da CNAE separados do campo misto.
- `snapshot_ano` marca o último mês de cada ano. `qtd_ativos` é estoque e não se soma entre meses.

In [0]:
%sql
CREATE OR REPLACE TABLE silver.cadastro_setor AS
SELECT
  CAST(ano AS INT)                                         AS ano,
  CAST(mes AS INT)                                         AS mes,
  make_date(CAST(ano AS INT), CAST(mes AS INT), 1)         AS data_ref,
  trim(categoria)                                          AS categoria,
  lpad(regexp_extract(cnae_fiscal, '^(\\d+)', 1), 7, '0')  AS cnae_subclasse,
  trim(regexp_replace(cnae_fiscal, '^\\d+\\s*-\\s*', ''))  AS nome_cnae_subclasse,
  regexp_extract(cnae_divisao, '^(\\d+)', 1)               AS cod_cnae_divisao,
  trim(regexp_replace(cnae_divisao, '^\\d+\\s*-\\s*', '')) AS nome_cnae_divisao,
  trim(atividade)                                          AS atividade,
  trim(area)                                               AS area,
  trim(setor)                                              AS setor,
  CAST(estabelecimentos_ativos AS INT)                     AS qtd_ativos,
  CAST(estabelecimentos_baixados AS INT)                   AS qtd_baixados,
  CAST(estabelecimentos_novos AS INT)                      AS qtd_novos,
  CAST(mes AS INT) = max(CAST(mes AS INT)) OVER (PARTITION BY CAST(ano AS INT)) AS snapshot_ano,
  _ingestao_ts,
  _fonte,
  current_timestamp()                                      AS _processamento_ts
FROM bronze.cadastro_contribuintes_setor;

In [0]:
%sql
-- Verificação: categorias do cadastro, que definem a proxy de micro e pequena empresa
SELECT categoria,
       count(*)                                                          AS linhas,
       sum(CASE WHEN ano = 2025 AND snapshot_ano THEN qtd_ativos END)    AS ativos_dez_2025
FROM silver.cadastro_setor
GROUP BY categoria ORDER BY ativos_dez_2025 DESC;

In [0]:
%sql
-- Verificação: linhas sem CNAE (a fonte publica '0000000 - SEM CNAE')
SELECT count(*) AS sem_cnae_na_fonte
FROM silver.cadastro_setor
WHERE cnae_subclasse = '0000000';

## 5. Cadastro de contribuintes por município

Grão: ano × mês × categoria × município. Mesma regra de estoque da seção 4.

In [0]:
%sql
CREATE OR REPLACE TABLE silver.cadastro_municipio AS
SELECT
  CAST(ano AS INT)                                             AS ano,
  CAST(mes AS INT)                                             AS mes,
  make_date(CAST(ano AS INT), CAST(mes AS INT), 1)             AS data_ref,
  CAST(cod_categ AS INT)                                       AS cod_categoria,
  trim(categoria)                                              AS categoria,
  lpad(regexp_replace(cod_municipio_ibge, '\\D', ''), 7, '0')  AS cod_municipio_ibge,
  trim(municipio_ibge)                                         AS nome_municipio,
  CAST(cod_corede AS INT)                                      AS cod_corede,
  upper(trim(corede))                                          AS nome_corede,
  CAST(qtd_ativos AS INT)                                      AS qtd_ativos,
  CAST(qtd_baixados AS INT)                                    AS qtd_baixados,
  CAST(qtd_novos AS INT)                                       AS qtd_novos,
  CAST(mes AS INT) = max(CAST(mes AS INT)) OVER (PARTITION BY CAST(ano AS INT)) AS snapshot_ano,
  _ingestao_ts,
  _fonte,
  current_timestamp()                                          AS _processamento_ts
FROM bronze.cadastro_contribuintes_municipio;

In [0]:
%sql
-- Verificação
SELECT count(DISTINCT cod_municipio_ibge) AS municipios,
       count(DISTINCT nome_corede)        AS coredes,
       min(ano) AS ano_min, max(ano) AS ano_max
FROM silver.cadastro_municipio;

In [0]:
%sql
SELECT ano,
       count(DISTINCT mes)                AS meses,
       count(DISTINCT cod_municipio_ibge) AS municipios,
       count(DISTINCT categoria)          AS categorias,
       sum(CASE WHEN snapshot_ano THEN qtd_ativos END) AS ativos_fim_de_ano
FROM mvp_pipeline_vf.silver.cadastro_municipio
GROUP BY ano ORDER BY ano;

## 6. De-para CNAE × cadeia produtiva

Grão: par subclasse × cadeia. Relação N:N: 36 subclasses pertencem a duas cadeias prioritárias.
`lpad` corrige 168 códigos sem zero à esquerda.

In [0]:
%sql
CREATE OR REPLACE TABLE silver.cnae_cadeia AS
SELECT DISTINCT
  lpad(regexp_replace(cnae_origem, '\\D', ''), 7, '0') AS cnae_subclasse,
  trim(denominacao_cnae)                               AS nome_cnae_subclasse,
  trim(cadeia)                                         AS cadeia,
  trim(cadeia) <> 'Outros (não priorizado)'            AS cadeia_prioritaria,
  _ingestao_ts,
  _fonte,
  current_timestamp()                                  AS _processamento_ts
FROM bronze.cnae_cadeia_produtiva
WHERE cnae_origem IS NOT NULL;

In [0]:
%sql
-- Verificação: distribuição de subclasses por cadeia
SELECT cadeia, count(*) AS cnaes
FROM silver.cnae_cadeia GROUP BY cadeia ORDER BY cnaes DESC;

In [0]:
%sql
-- Verificação: subclasses em mais de uma cadeia prioritária (esperado 36)
--              e códigos fora do padrão de 7 dígitos (esperado 0)
SELECT
  (SELECT count(*) FROM (
     SELECT cnae_subclasse FROM silver.cnae_cadeia WHERE cadeia_prioritaria
     GROUP BY cnae_subclasse HAVING count(DISTINCT cadeia) > 1)) AS cnaes_em_duas_cadeias,
  (SELECT count(*) FROM silver.cnae_cadeia
   WHERE length(cnae_subclasse) <> 7)                            AS fora_do_padrao;

## 7. Municípios do IBGE

Grão: município. Fornece microrregião e mesorregião.

In [0]:
%sql
CREATE OR REPLACE TABLE silver.municipio_ibge AS
SELECT
  lpad(CAST(id AS STRING), 7, '0') AS cod_municipio_ibge,
  trim(nome)                       AS nome_municipio,
  microrregiao.nome                AS microrregiao,
  microrregiao.mesorregiao.nome    AS mesorregiao,
  current_timestamp()              AS _processamento_ts
FROM bronze.ibge_municipios_rs;

## 8. PIB municipal

Grão: ano × município. Recorte 2018–2023.

**Fonte.** PIB e PIB per capita do DEE-RS, conveniado do IBGE no cálculo do PIB municipal. A série do
IBGE foi comparada município a município: diferença de 5 partes por bilhão, explicada pela unidade
de publicação (IBGE em mil reais, DEE em reais). Adotado o DEE por coerência interna, precisão e
economia do pipeline.

**População.** Derivada da razão entre PIB e PIB per capita, ambos do mesmo cálculo. Erro de
arredondamento inferior a uma pessoa: Porto Alegre, 2022, resulta em 1.332.569 contra 1.332.570 do
Censo.

**Quebra da série.** Até 2021 a população é estimativa do IBGE; de 2022 em diante, Censo 2022. O
estado passa de 11,47 para 10,88 milhões de habitantes. A coluna `origem_populacao` marca a quebra.
O PIB per capita é comparável entre municípios no mesmo ano, não entre 2021 e 2022 no mesmo
município.

In [0]:
%sql
CREATE OR REPLACE TABLE silver.pib_municipal AS
WITH dee AS (
  SELECT
    lpad(cod_municipio_ibge, 7, '0')                                 AS cod_municipio_ibge,
    trim(nome_municipio)                                             AS nome_municipio,
    CAST(ano AS INT)                                                 AS ano,
    CAST(regexp_replace(pib, '\\.', '') AS DECIMAL(18,2))            AS pib_reais,
    CAST(regexp_replace(pib_per_capita, '\\.', '') AS DECIMAL(18,2)) AS pib_per_capita_reais
  FROM bronze.dee_pib_municipal_rs
  WHERE pib_per_capita <> ''
    AND CAST(ano AS INT) BETWEEN 2018 AND 2023
)
SELECT
  cod_municipio_ibge,
  nome_municipio,
  ano,
  pib_reais,
  pib_per_capita_reais,
  CAST(round(pib_reais / pib_per_capita_reais) AS INT)                AS populacao,
  CASE WHEN ano <= 2021 THEN 'Estimativa IBGE' ELSE 'Censo 2022' END  AS origem_populacao,
  current_timestamp()                                                 AS _processamento_ts
FROM dee;

In [0]:
%sql
-- Verificação: perfil da série
SELECT
  ano,
  count(*)                     AS municipios,
  sum(pib_reais)               AS pib_total,
  count(pib_per_capita_reais)  AS com_per_capita,
  sum(populacao)               AS populacao_rs
FROM silver.pib_municipal GROUP BY ano ORDER BY ano;

In [0]:
%sql
-- Verificação: coerência entre as três medidas e cobertura do código IBGE
SELECT
  (SELECT count(*) FROM silver.pib_municipal
   WHERE abs(pib_reais / populacao - pib_per_capita_reais) > 1)      AS linhas_incoerentes,
  (SELECT count(*) FROM silver.municipio_ibge)                       AS municipios_ibge,
  (SELECT count(*) FROM (
     SELECT DISTINCT cod_municipio_ibge FROM silver.pib_municipal) d
   LEFT ANTI JOIN silver.municipio_ibge m
     ON m.cod_municipio_ibge = d.cod_municipio_ibge)                 AS pib_fora_do_ibge;

## 9. De-para de município (SEFAZ × IBGE)

A SEFAZ usa código próprio de município. A correspondência com o IBGE é feita pelo nome normalizado:
maiúsculas, sem acento, sem caractere não alfanumérico.

Os códigos 0 ("Sem Município") e 900 ("Outras UF") recebem `pseudo_municipio = TRUE`. Somam no
total do estado e ficam fora da análise territorial.

In [0]:
%sql
CREATE OR REPLACE TABLE silver.depara_municipio AS
WITH sefaz AS (
  SELECT DISTINCT cod_munic_sefaz,
         nome_municipio,
         regexp_replace(
           translate(upper(nome_municipio),
                     'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ',
                     'AAAAAEEEEIIIIOOOOOUUUUC'),
           '[^A-Z0-9]', '') AS chave
  FROM silver.arrecadacao_municipio
),
ibge AS (
  SELECT DISTINCT cod_municipio_ibge,
         nome_municipio AS nome_ibge,
         regexp_replace(
           translate(upper(nome_municipio),
                     'ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ',
                     'AAAAAEEEEIIIIOOOOOUUUUC'),
           '[^A-Z0-9]', '') AS chave
  FROM silver.cadastro_municipio
)
SELECT s.cod_munic_sefaz,
       s.nome_municipio,
       i.cod_municipio_ibge,
       i.nome_ibge,
       s.cod_munic_sefaz IN (0, 900) AS pseudo_municipio,
       current_timestamp() AS _processamento_ts
FROM sefaz s
LEFT JOIN ibge i ON s.chave = i.chave;

In [0]:
%sql
-- Verificação: nenhum município real pode ficar sem código IBGE
SELECT count(*) AS sem_correspondencia
FROM silver.depara_municipio
WHERE cod_municipio_ibge IS NULL AND NOT pseudo_municipio;

## 10. Verificação final da camada

In [0]:
%sql
SELECT 'icms_cnae_subclasse' AS tabela, count(*) AS linhas FROM silver.icms_cnae_subclasse
UNION ALL SELECT 'arrecadacao_municipio', count(*) FROM silver.arrecadacao_municipio
UNION ALL SELECT 'desoneracoes',          count(*) FROM silver.desoneracoes
UNION ALL SELECT 'cadastro_setor',        count(*) FROM silver.cadastro_setor
UNION ALL SELECT 'cadastro_municipio',    count(*) FROM silver.cadastro_municipio
UNION ALL SELECT 'cnae_cadeia',           count(*) FROM silver.cnae_cadeia
UNION ALL SELECT 'municipio_ibge',        count(*) FROM silver.municipio_ibge
UNION ALL SELECT 'pib_municipal',         count(*) FROM silver.pib_municipal
UNION ALL SELECT 'depara_municipio',      count(*) FROM silver.depara_municipio
ORDER BY tabela;